In [1]:
%pip install -q groq

In [2]:

import json, functools
from getpass import getpass
from groq import Groq, RateLimitError, APIConnectionError

client = Groq(api_key=getpass("Enter Groq API Key: "))

MODEL = "openai/gpt-oss-120b"

MENU = {
    "veg burger": 100,
    "chicken burger": 150,
    "margherita pizza": 200,
    "farmhouse pizza": 280,
    "white sauce pasta": 180,
    "french fries": 80,
    "cold coffee": 90,
    "chocolate milkshake": 120
}

SYSTEM = f"""ROLE: You are FoodMate, a friendly restaurant ordering assistant.
FACTS: Menu with prices in rupees: {json.dumps(MENU)}
RULES:
1. Help customers explore the menu and place orders.
2. Remember customer names and order details.
3. Never invent menu items or prices.
4. Use available Python functions for price and cart operations.
5. Allow customers to add, remove and update cart items.
6. Show the complete bill before order confirmation.
7. Never claim an order is placed without customer confirmation.
8. If information is unavailable, say "I'll check with the restaurant."
FORMAT: Under 80 words, friendly English."""

cart = {}

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price for an item."""
    item = item.lower().strip()
    if item not in MENU or qty <= 0:
        raise ValueError("Invalid item or quantity")
    return MENU[item] * qty

@log_call
def add_to_cart(item: str, qty: int = 1) -> str:
    item = item.lower().strip()
    if item not in MENU or qty <= 0:
        return "Invalid item or quantity"
    cart[item] = cart.get(item, 0) + qty
    return f"Added {qty} {item}(s) to cart."

@log_call
def remove_from_cart(item: str) -> str:
    item = item.lower().strip()
    if item not in cart:
        return "Item not found in cart."
    del cart[item]
    return f"Removed {item} from cart."

@log_call
def update_quantity(item: str, qty: int) -> str:
    item = item.lower().strip()
    if item not in cart or qty <= 0:
        return "Invalid item or quantity."
    cart[item] = qty
    return f"Updated {item} quantity to {qty}."

@log_call
def view_cart() -> str:
    if not cart:
        return "Your cart is empty."
    details = [
        f"{item}: {qty} x ₹{MENU[item]} = ₹{MENU[item] * qty}"
        for item, qty in cart.items()
    ]
    total = sum(MENU[item] * qty for item, qty in cart.items())
    details.append(f"Total: ₹{total}")
    return "\n".join(details)

@log_call
def calculate_bill() -> int:
    return sum(MENU[item] * qty for item, qty in cart.items())


# Tool definitions

def make_tool(name, description, properties, required):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
                "additionalProperties": False
            }
        }
    }

TOOLS = [
    make_tool("get_price", "Calculate item price",
        {"item": {"type": "string", "enum": list(MENU)},
         "qty": {"type": "integer"}}, ["item", "qty"]),

    make_tool("add_to_cart", "Add item to cart",
        {"item": {"type": "string", "enum": list(MENU)},
         "qty": {"type": "integer"}}, ["item", "qty"]),

    make_tool("remove_from_cart", "Remove item from cart",
        {"item": {"type": "string", "enum": list(MENU)}}, ["item"]),

    make_tool("update_quantity", "Update cart quantity",
        {"item": {"type": "string", "enum": list(MENU)},
         "qty": {"type": "integer"}}, ["item", "qty"]),

    make_tool("view_cart", "Display cart", {}, []),

    make_tool("calculate_bill", "Calculate total bill", {}, [])
]

FUNCTIONS = {
    "get_price": get_price,
    "add_to_cart": add_to_cart,
    "remove_from_cart": remove_from_cart,
    "update_quantity": update_quantity,
    "view_cart": view_cart,
    "calculate_bill": calculate_bill
}

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8


def chat(user_text: str) -> str:

    history.append({"role": "user", "content": user_text})

    try:
        messages = [history[0]] + history[1:][-MAX_TURNS:]

        for _ in range(5):

            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=TOOLS,
                tool_choice="auto",
                temperature=0.3
            )

            message = response.choices[0].message

            if not message.tool_calls:
                reply = message.content or "How else can I help?"
                history.append({"role": "assistant", "content": reply})
                return reply

            assistant_message = message.model_dump(exclude_none=True)
            messages.append(assistant_message)
            history.append(assistant_message)

            for call in message.tool_calls:

                name = call.function.name
                args = json.loads(call.function.arguments)

                try:
                    if name not in FUNCTIONS:
                        result = "Unknown function."
                    else:
                        result = FUNCTIONS[name](**args)

                except (ValueError, TypeError, KeyError) as e:
                    result = str(e)

                tool_result = {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": str(result)
                }

                messages.append(tool_result)
                history.append(tool_result)

        reply = "Please try again."

    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."

    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."

    except Exception as e:
        print("Error:", e)
        reply = "Something went wrong. Please try again."

    history.append({"role": "assistant", "content": reply})
    return reply


print("Welcome to FoodMate Restaurant!")
print("Type 'bye' to exit.\n")

while True:
    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting FoodMate!")
        break

    print("Bot:", chat(text))


Enter Groq API Key: ··········
Welcome to FoodMate Restaurant!
Type 'bye' to exit.

You: Show me the menu
Bot: Sure! Here’s our menu (prices in ₹):

- Veg Burger – 100  
- Chicken Burger – 150  
- Margherita Pizza – 200  
- Farmhouse Pizza – 280  
- White Sauce Pasta – 180  
- French Fries – 80  
- Cold Coffee – 90  
- Chocolate Milkshake – 120  

Let me know what you’d like to add to your cart!
You: Add 2 chicken burgers
   [log] add_to_cart() -> 'Added 2 chicken burger(s) to cart.'
Bot: Great! I’ve added 2 Chicken Burgers to your cart. Anything else you’d like to add or adjust?
You: Add one cold coffee
   [log] add_to_cart() -> 'Added 1 cold coffee(s) to cart.'
   [log] view_cart() -> 'chicken burger: 2 x ₹150 = ₹300\ncold coffee: 1 x ₹90 = ₹90\nTotal: ₹390'
Bot: Your cart now has:

- Chicken Burger × 2 = ₹300  
- Cold Coffee × 1 = ₹90  

**Subtotal:** ₹390  

Would you like to add anything else, update quantities, or see the final bill?
You: Show my cart
   [log] view_cart() -> 'chi